# 🍎 Part A: Few-Shot Classification with Transfer Learning
**Student:** Anderson David Arenas Gutiérrez  
**Course:** Machine Learning - Reto 7  
**Instructor:** Carlos Andrés Sierra, M.Sc.  

---

In [ ]:
import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch_directml
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms, models

# Configure hardware acceleration (AMD GPU)
device = torch_directml.device(1) if torch_directml.is_available() else torch.device('cpu')
print(f"🚀 Device: {device}")

SEEDS = [42, 100, 2026]
BASE_DIR = "C:\\Users\\Anderson\\Documents\\UD\\7mo\\MachineLearning\\Challenges\\challenge-7_5"
DATA_DIR = os.path.join(BASE_DIR, "data")
LOG_DIR = os.path.join(BASE_DIR, "runs")
CKPT_DIR = os.path.join(BASE_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

imsize = 224
batch_size = 32

transform_train = transforms.Compose([
    transforms.Resize((imsize, imsize)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((imsize, imsize)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

### Multi-seed Training Loop - Real Source Phase

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_r, corr, tot = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        loss_r += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        corr += torch.sum(preds == labels.data)
        tot += inputs.size(0)
    return loss_r / tot, (corr.double() / tot).item()

source_train_dir = os.path.join(DATA_DIR, "source_real", "train")
target_val_dir = os.path.join(DATA_DIR, "target_infograph", "test")
dataset_val = datasets.ImageFolder(target_val_dir, transform=transform_val)
loader_val = DataLoader(dataset_val, batch_size=batch_size, shuffle=False)

for seed in SEEDS:
    print(f"🌱 Running Seed {seed} for Part A")
    set_seed(seed)
    dataset_source = datasets.ImageFolder(source_train_dir, transform=transform_train)
    loader_source = DataLoader(dataset_source, batch_size=batch_size, shuffle=True)
    
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, len(dataset_source.classes))
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    writer = SummaryWriter(os.path.join(LOG_DIR, f"seed_{seed}_M1_feat_extract"))
    
    epochs = 25
    for epoch in range(1, epochs + 1):
        loss_t, acc_t = train_epoch(model, loader_source, criterion, optimizer)
        writer.add_scalar("Loss/Train", loss_t, epoch)
        writer.add_scalar("Accuracy/Train", acc_t, epoch)
        
    if seed == 42:
        torch.save(model.state_dict(), os.path.join(CKPT_DIR, "best_part_A_model.pt"))
        print("💾 Part A checkpoint saved successfully.")
    writer.close()